# AI Gimbal Camera — Live Demo

Run this notebook to test the full system. Works with any USB or built-in webcam.

**Controls:** Buttons below the video feed.

In [ ]:
import sys, os, time, threading, cv2, numpy as np
from IPython.display import display, clear_output
import ipywidgets as widgets

# Add project root to path
sys.path.append('..')

from src.capture.camera import Camera
from src.cv.face_detector import FaceDetector, FaceTracker
from src.cv.gaze_estimator import GazeEstimator
from src.cv.gesture_classifier import GestureClassifier
from src.control.gimbal import GimbalController
from src.control.pid import PIDController
from src.control.state_machine import StateMachine, Mode
from src.utils.config import load_config
from src.utils.visualization import (
    draw_debug_overlay, compute_framing_error, crop_face_region
)

print('All imports loaded successfully')

In [ ]:
# Load config
cfg = load_config('../config/default.yaml')

# Initialize all components
camera = Camera(source=0, width=640, height=480)
face_detector = FaceDetector(min_confidence=0.5)
face_tracker = FaceTracker(max_lost_frames=5)
gaze_estimator = GazeEstimator(
    l2cs_path='../models/l2cs_net.pth' if os.path.exists('../models/l2cs_net.pth') else None,
    custom_cnn_path='../models/gaze_cnn.pth' if os.path.exists('../models/gaze_cnn.pth') else None
)
gesture_classifier = GestureClassifier(
    svm_path='../models/gesture_svm.pkl' if os.path.exists('../models/gesture_svm.pkl') else None,
    scaler_path='../models/gesture_scaler.pkl' if os.path.exists('../models/gesture_scaler.pkl') else None
)

# Gimbal (runs in mock mode if serial not available)
try:
    gimbal = GimbalController(port=cfg.serial.port, baud=cfg.serial.baud)
    gimbal.home()
    print(f'Gimbal connected on {cfg.serial.port}')
except Exception as e:
    print(f'Gimbal not connected: {e}')
    print('Running in mock mode (angles printed only)')
    gimbal = None

pid_pan = PIDController(**cfg.pid.pan.__dict__)
pid_tilt = PIDController(**cfg.pid.tilt.__dict__)
state_machine = StateMachine(idle_timeout_frames=cfg.state_machine.idle_timeout_frames)

print('All components initialized')

In [ ]:
# Widgets
image_widget = widgets.Image(format='jpg', width=640, height=480)
fps_label = widgets.HTML(value='FPS: --')
mode_label = widgets.HTML(value='Mode: IDLE')
gaze_label = widgets.HTML(value='Gaze: --')
gesture_label = widgets.HTML(value='Gesture: --')

btn_quit = widgets.Button(description='Quit', button_style='danger')
btn_home = widgets.Button(description='Home', button_style='primary')
btn_lock = widgets.Button(description='Lock/Unlock', button_style='warning')
btn_record = widgets.Button(description='Record', button_style='info')

control_box = widgets.HBox([btn_quit, btn_home, btn_lock, btn_record])
info_box = widgets.HBox([fps_label, mode_label, gaze_label, gesture_label])
ui = widgets.VBox([image_widget, info_box, control_box])

display(ui)

In [ ]:
# State variables
running = True
recording = False
frame_count = 0
fps_timer = time.time()
fps_counter = 0
current_fps = 0.0

# Button callbacks
def on_quit(b):
    global running
    running = False

def on_home(b):
    global gimbal, state_machine
    if gimbal:
        gimbal.home()
    state_machine.mode = Mode.HOME

def on_lock(b):
    state_machine.toggle_lock()

def on_record(b):
    global recording
    recording = not recording
    btn_record.description = 'Stop' if recording else 'Record'
    btn_record.button_style = 'danger' if recording else 'info'

btn_quit.on_click(on_quit)
btn_home.on_click(on_home)
btn_lock.on_click(on_lock)
btn_record.on_click(on_record)

print('Ready. Press Play button below to start.')

In [ ]:
# Main loop
running = True
frame_count = 0
fps_timer = time.time()
fps_counter = 0
current_fps = 0.0

while running:
    frame_obj = camera.read()
    if frame_obj is None:
        continue

    frame = frame_obj.data
    frame_count += 1
    fps_counter += 1
    if time.time() - fps_timer >= 1.0:
        current_fps = fps_counter / (time.time() - fps_timer)
        fps_counter = 0
        fps_timer = time.time()

    h, w = frame.shape[:2]

    # Face detection
    faces = face_detector.detect(frame)
    face = face_tracker.update(faces)
    state_machine.update_face_status(face is not None)

    # PID control
    if state_machine.mode == Mode.TRACKING and face:
        error_x, error_y = compute_framing_error(
            face.bbox, (w, h), cfg.pid.dead_zone
        )
        delta_pan = pid_pan.update(error_x)
        delta_tilt = pid_tilt.update(error_y)
        if gimbal:
            gimbal.set_pan_delta(delta_pan)
            gimbal.set_tilt_delta(delta_tilt)
    elif state_machine.mode == Mode.IDLE:
        pid_pan.reset()
        pid_tilt.reset()

    # Gesture + Gaze (every 6th frame)
    gesture_result = None
    gaze_result = None
    if frame_count % 6 == 0:
        gesture_result = gesture_classifier.predict(frame)
        if gesture_result:
            state_machine.process_gesture(gesture_result.gesture)

        if face:
            face_crop = crop_face_region(frame, face.bbox)
            gaze_result = gaze_estimator.predict(face_crop)
            if gaze_result:
                state_machine.update_gaze(gaze_result.direction)

    # Overlay
    pan_angle = gimbal.pan_angle if gimbal else 90
    tilt_angle = gimbal.tilt_angle if gimbal else 90
    overlay = draw_debug_overlay(
        frame=frame, face=face, gaze=gaze_result,
        gesture=gesture_result, mode=state_machine.mode,
        fps=current_fps, recording=recording,
        gimbal_angles=(pan_angle, tilt_angle),
    )

    # Update widgets
    _, jpeg = cv2.imencode('.jpg', overlay)
    image_widget.value = jpeg.tobytes()
    fps_label.value = f'FPS: {current_fps:.1f}'
    mode_label.value = f'Mode: {state_machine.mode.value}'
    if gaze_result:
        labels = ['CENTER', 'LEFT', 'RIGHT', 'UP', 'DOWN']
        gaze_label.value = f'Gaze: {labels[gaze_result.direction]}'
    if gesture_result:
        gesture_label.value = f'Gesture: {gesture_result.gesture}'

print('Demo stopped.')

In [ ]:
# Cleanup
if gimbal:
    gimbal.home()
camera.release()
print('Cleanup done. You can re-run the setup cell to restart.')